In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub

# Huggingface login

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
huggingface_api = user_secrets.get_secret("huggingface_api")
login(token=huggingface_api)

# Utils

In [ ]:
import time
import json
import torch
import re
import os
from tqdm import tqdm
import transformers
from transformers import AutoModelForCausalLM, Gemma3ForConditionalGeneration, AutoProcessor, AutoTokenizer, pipeline
from typing import Dict, Any, List, Tuple
from datetime import datetime
import sys
import gc


## Random seed setting

In [4]:
import torch
import numpy as np
import random
import os

def setup_reproducible_environment(seed: int = 42):
    """
    Setup reproducible environment for scientific experiments.
    
    Args:
        seed: Random seed for reproducibility
        
    Note:
        This function ensures deterministic behavior across runs
        for scientific reproducibility requirements.
    """
    # Python random
    random.seed(seed)
    
    # NumPy random
    np.random.seed(seed)
    
    # PyTorch random
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # For multi-GPU
    
    # Ensure deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Set environment variable for additional determinism
    os.environ['PYTHONHASHSEED'] = str(seed)

setup_reproducible_environment()

## Prompt Technical

In [5]:
def create_zero_shot_prompt(text, sent_id) -> Tuple[str, str]:
    """
    Tạo prompt cho zero-shot sentiment analysis với định dạng đầu ra cụ thể.
    
    Args:
        text: Vietnamese text to analyze
        sent_id: Unique identifier for the sentence
        
    Returns:
        Tuple of (system_prompt, user_prompt) for model input
        
    Note:
        System prompt contains detailed instructions for Vietnamese
        structured sentiment analysis with JSON output format.
    """
    system_prompt = """Bạn là chuyên gia phân tích cảm xúc tiếng Việt có cấu trúc. Nhiệm vụ của bạn là phân tích bình luận mạng xã hội và trích xuất các thành phần cảm xúc theo cấu trúc JSON.

ĐỊNH NGHĨA CÁC THÀNH PHẦN:

1. SOURCE (Nguồn gốc bình luận):
   - Người phát biểu ý kiến, có thể là người bình luận hoặc được trích dẫn
   - Thường là các đại từ nhân xưng: "Tôi", "Tao", "Mình", "Bọn tao", "Mẹ tui"
   - Có thể có hoặc không có trong câu

2. TARGET (Đối tượng hướng tới):
   - Cá nhân, tập thể, sự vật, hiện tượng mà bình luận hướng đến
   - Thường là các đại từ xưng hô: "Mày", "Cậu", "Anh ấy", "Bạn"
   - Có thể có hoặc không có trong câu

3. POLAR_EXPRESSION (Biểu thức cảm xúc):
   - Từ/cụm từ bày tỏ cảm xúc, ý nghĩ, cảm nhận, hành động
   - Bao gồm: tính từ cảm xúc, thán từ, hành động xúc phạm/khen ngợi
   - Ví dụ: "buồn", "vui", "tức giận", "đáng đời", "đánh"
   - BẮT BUỘC phải có

4. POLARITY (Tính chất cảm xúc):
   - Positive: Khích lệ, động viên, chia sẻ, vui đùa không xúc phạm
   - Negative: Xúc phạm, kích động, chia rẽ, gây thù ghét
   - Neutral: Bình luận bình thường, khách quan

5. INTENSITY (Cường độ cảm xúc):
   - Strong: Cảm xúc mạnh mẽ, từ ngữ quyết liệt
   - Standard: Cảm xúc bình thường, từ ngữ thông thường  
   - Weak: Cảm xúc nhẹ nhàng, từ ngữ dè dặt

QUY TẮC PHÂN TÍCH:
- Mỗi câu có thể chứa nhiều opinion khác nhau
- Mỗi opinion phải có ít nhất 1 Polar_expression
- Polar_expression là thành phần bắt buộc phải có
- Chú ý các từ viết tắt, teencode, hàm ý, ẩn ý trong tiếng Việt

FORMAT JSON OUTPUT:
{
  "text": "[Bình luận gốc]",
  "opinions": [
    {
      "Source": ["text_span_1", "text_span_2"],
      "Target": ["text_span_1", "text_span_2"],
      "Polar_expression": ["text_span_1", "text_span_2"],
      "Polarity": "Positive/Negative/Neutral",
      "Intensity": "Strong/Standard/Weak"
    }
  ]
}

QUAN TRỌNG: CHỈ TRẢ VỀ JSON, KHÔNG GIẢI THÍCH THÊM, kHÔNG TRÌNH BÀY QUÁ TRÌNH SUY LUẬN.
"""

    user_prompt = f"""Phân tích cảm xúc cho văn bản sau (sent_id: {sent_id}):
"{text}"

Trả về KẾT QUẢ CHÍNH XÁC theo cấu trúc JSON đã yêu cầu."""
    return system_prompt, user_prompt

## Extract information from respone

In [6]:
def extract_position(text, expression) -> str:
    """
    Trích xuất vị trí bắt đầu và kết thúc của biểu thức trong văn bản.
    
    Args:
        text: Original text
        expression: Expression to find position for
        
    Returns:
        Position string in format "start:end" (0-indexed)
        
    Example:
        extract_position("Tôi rất vui", "rất vui") -> "4:12"
    """
    start = text.find(expression)
    if start == -1:
        return "0:0"
    end = start + len(expression)
    return f"{start}:{end}"

def postprocess_response(response_text, original_text, sent_id):
    """
    Chuẩn hóa kết quả trả về từ model để phù hợp với định dạng SemEval.
    
    Args:
        response_text: Raw JSON response from model
        original_text: Original input text
        sent_id: Sentence identifier
        
    Returns:
        Formatted JSON string with validated structure and auto-extracted positions
        
    Note:
        Handles new simplified format where components are arrays of text spans.
        Automatically extracts positions for all text spans.
    """
    try:
        result = json.loads(response_text)
        result["sent_id"] = sent_id
        result["text"] = original_text
        
        if "opinions" in result and isinstance(result["opinions"], list):
            for opinion in result["opinions"]:
                if not isinstance(opinion, dict): 
                    continue
                
                # Process each component: Source, Target, Polar_expression
                for component in ["Source", "Target", "Polar_expression"]:
                    if component not in opinion:
                        opinion[component] = [[], []]
                        continue
                    
                    component_data = opinion[component]
                    
                    # Handle new simplified format: array of text spans
                    if isinstance(component_data, list):
                        # Check if already in [texts, positions] format
                        if (len(component_data) == 2 and 
                            isinstance(component_data[0], list) and 
                            isinstance(component_data[1], list)):
                            # Old format - re-extract positions anyway for accuracy
                            texts = component_data[0]
                        else:
                            # New format - just array of text spans
                            texts = component_data
                        
                        # Clean and extract positions
                        valid_texts = [text for text in texts if isinstance(text, str) and text.strip()]
                        positions = [extract_position(original_text, text) for text in valid_texts]
                        opinion[component] = [valid_texts, positions]
                        
                    elif isinstance(component_data, str) and component_data.strip():
                        # Single string
                        text = component_data.strip()
                        position = extract_position(original_text, text)
                        opinion[component] = [[text], [position]]
                    else:
                        # Empty or invalid
                        opinion[component] = [[], []]
                
                # Set proper defaults
                if "Polarity" not in opinion or opinion["Polarity"] not in ["Positive", "Negative", "Neutral"]:
                    opinion["Polarity"] = ""
                    
                if "Intensity" not in opinion or opinion["Intensity"] not in ["Strong", "Standard", "Weak"]:
                    opinion["Intensity"] = ""
        else:
            result["opinions"] = []
            
        return json.dumps(result, ensure_ascii=False, indent=2)
        
    except json.JSONDecodeError:
        default_result = {"sent_id": sent_id, "text": original_text, "opinions": []}
        return json.dumps(default_result, ensure_ascii=False, indent=2)
    
def extract_json_from_response(response: str) -> str:
    """
    Trích xuất phần JSON từ response của model với support đa dạng format.
    
    Args:
        response: Raw model response text
        
    Returns:
        Extracted JSON string or cleaned response
        
    Note:
        Handles various response formats from different models:
        - Gemma: Assistant markers
        - Qwen: Thinking tokens  
        - Mistral/DeepSeek: Standard JSON formats
        - Universal: Backticks, greedy matching, fallbacks
    """
    original_response = response
    
    # STAGE 1: Remove model-specific markers
    
    # 1.1: Handle Gemma assistant markers
    assistant_markers = ["assistant:", "assistant", "<assistant>"]
    for marker in assistant_markers:
        if marker in response:
            response = response.split(marker, 1)[1].strip()
    
    # 1.2: Handle Qwen thinking tokens
    # Remove <think>...</think> blocks
    think_block_pattern = r"<think>[\s\S]*?</think>\s*"
    match = re.match(think_block_pattern, response, re.DOTALL)
    if match:
        response = response[match.end():].strip()
    
    # Handle <|thought|> marker
    thought_marker = "<|thought|>"
    if thought_marker in response:
        parts = response.split(thought_marker)
        response = parts[-1].strip()
    
    # STAGE 2: Extract JSON with multiple strategies
    
    # 2.1: Find JSON within triple backticks (most reliable)
    match_backticks = re.search(r'```json\s*([\s\S]*?)\s*```', response, re.DOTALL)
    if match_backticks:
        json_candidate = match_backticks.group(1).strip()
        try:
            json.loads(json_candidate)
            return json_candidate
        except json.JSONDecodeError:
            pass
    
    # 2.2: Find complete JSON object with brace matching (Qwen approach)
    first_brace_index = response.find('{')
    if first_brace_index != -1:
        open_braces = 0
        for i in range(first_brace_index, len(response)):
            if response[i] == '{':
                open_braces += 1
            elif response[i] == '}':
                open_braces -= 1
                if open_braces == 0:
                    json_candidate = response[first_brace_index : i+1]
                    try:
                        json.loads(json_candidate)
                        return json_candidate.strip()
                    except json.JSONDecodeError:
                        break
    
    # 2.3: Greedy JSON matching (Mistral/DeepSeek approach)
    json_pattern_greedy = r'({[\s\S]*})'
    match_greedy = re.search(json_pattern_greedy, response)
    if match_greedy:
        json_candidate_greedy = match_greedy.group(1).strip()
        try:
            json.loads(json_candidate_greedy)
            return json_candidate_greedy
        except json.JSONDecodeError:
            pass
    
    # 2.4: Find all potential JSON objects and test (fallback)
    json_pattern_findall = r'({[\s\S]*?})'
    json_matches = re.findall(json_pattern_findall, response)
    if json_matches:
        # Try from largest to smallest (reversed order)
        for match_str in reversed(json_matches):
            candidate = match_str.strip()
            try:
                json.loads(candidate)
                return candidate
            except json.JSONDecodeError:
                continue
    
    # STAGE 3: Final fallback
    return response.strip()

## Other functions

In [7]:
def save_experiment_metadata(
    config: Dict[str, Any], 
    results_summary: Dict[str, Any], 
    output_dir: str = "/kaggle/working"
) -> str:
    """
    Save experiment metadata for reproducibility tracking.
    
    Args:
        config: Experiment configuration parameters
        results_summary: Summary of experiment results
        output_dir: Directory to save metadata
        
    Returns:
        Path to saved metadata file
        
    Example:
        config = {
            "model_id": "google/gemma-3-12b-it",
            "batch_size": 8,
            "max_tokens": 512
        }
        save_experiment_metadata(config, results)
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Get GPU info if available
    gpu_info = {}
    if torch.cuda.is_available():
        gpu_info = {
            "gpu_count": torch.cuda.device_count(),
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "Unknown",
            "cuda_version": torch.version.cuda
        }
    
    metadata = {
        "timestamp": timestamp,
        "experiment_config": config,
        "results_summary": results_summary,
        "environment": {
            "python_version": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
            "torch_version": torch.__version__,
            "transformers_version": transformers.__version__,
            "gpu_info": gpu_info
        }
    }
    
    # Save metadata
    metadata_file = os.path.join(output_dir, f"experiment_metadata_{timestamp}.json")
    with open(metadata_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    
    print(f"📊 Experiment metadata saved: {metadata_file}")
    return metadata_file

def load_dataset(dataset_path) -> List[Dict[str, Any]]:
    """Load dataset from JSON file"""
    try:
        with open(dataset_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"Successfully loaded {len(data)} samples from {dataset_path}")
        return data
    except Exception as e:
        print(f"Error loading dataset from {dataset_path}: {e}")
        return []
    
def gpu_memory_cleanup():
    """Clean up GPU memory - important for T4 with limited VRAM."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()
        # More aggressive cleanup for T4 GPUs
        for i in range(torch.cuda.device_count()):
            with torch.cuda.device(i):
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
    print("🧹 GPU memory cleanup completed")
        
def print_result(result_json, gen_time):
    """
    Hiển thị kết quả phân tích dạng JSON với định dạng đẹp.
    
    Args:
        result_json: JSON result string to display
        gen_time: Generation time in seconds
        
    Note:
        Handles both valid JSON and fallback raw output display.
    """
    try:
        result = json.loads(result_json)
        print("\n" + "="*50)
        print(f"KẾT QUẢ PHÂN TÍCH CẢM XÚC (thời gian: {gen_time:.2f}s):")
        print("-"*50)
        print(json.dumps(result, ensure_ascii=False, indent=2))
        print("="*50)
    except json.JSONDecodeError: # Changed from generic except
        print("\n" + "="*50)
        print(f"KẾT QUẢ PHÂN TÍCH CẢM XÚC (thời gian: {gen_time:.2f}s) - RAW OUTPUT (JSON PARSE FAILED):")
        print("-"*50)
        print(result_json)
        print("="*50)

# Qwen2.5-7B-Instruct

## Config

In [ ]:
# ================================
# QWEN 2.5 CONFIGURATION  
# ================================

# Model Configuration
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
TORCH_DTYPE = torch.float32  # Options: torch.float32, torch.float16, torch.bfloat16
DEVICE_MAP = "auto"  # Options: "auto", "cpu", specific GPU mapping
TRUST_REMOTE_CODE = True  # Required for Qwen models

# Generation Configuration  
MAX_TOKENS = 512  # Options: 256, 512, 1024, 2048
DO_SAMPLE = False  # Options: True, False
TEMPERATURE = 0.1  # Only used if DO_SAMPLE=True
TOP_P = 0.95      # Only used if DO_SAMPLE=True
TOP_K = 50        # Only used if DO_SAMPLE=True

# Processing Configuration
BATCH_SIZE = 4    # Options: 1, 2, 4, 8 (optimized for Qwen 2.5-7B on 2xT4)
CLEANUP_FREQUENCY = 10  # GPU cleanup after every N batches

# Qwen-Specific Configuration
PADDING_SIDE = "left"  # Qwen tokenizer preference
AUTO_PAD_TOKEN = True  # Auto-configure pad_token if missing

# Dataset Configuration
DATASET_PATH = "/kaggle/input/dev.json"
OUTPUT_FILE = "/kaggle/working/qwen25_results.json"

# Experiment Configuration
EXPERIMENT_NAME = "qwen25_baseline"

# ================================
# END CONFIGURATION
# ================================

## Inference

In [ ]:
"""LLM test performance with Qwen/Qwen2.5-7B-Instruct on Kaggle with 2xT4 GPUs"""

def setup_model() -> Tuple[AutoModelForCausalLM, AutoProcessor]:
    """
    Initialize the Qwen/Qwen2.5-7B-Instruct model and processor.
    
    Returns:
        Tuple of (model, processor) ready for inference
        
    Raises:
        RuntimeError: If model initialization fails
        
    Note:
        Uses device_map="auto" for multi-GPU distribution on 2xT4 setup.
        Uses float32 for better stability and compatibility.
    """
    model_id = "Qwen/Qwen2.5-7B-Instruct"
    print(f"Initializing model: {model_id}")
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map=DEVICE_MAP,
        dtype=TORCH_DTYPE,
        trust_remote_code=TRUST_REMOTE_CODE
    ).eval()

    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
    
    # Qwen tokenizer configuration
    if AUTO_PAD_TOKEN and processor.pad_token_id is None:
        processor.pad_token_id = processor.eos_token_id
        if model.config.pad_token_id is None:
            model.config.pad_token_id = processor.eos_token_id
    
    processor.padding_side = PADDING_SIDE
    print("Model and processor are ready.")
    return model, processor


def generate_prediction(
    model: AutoModelForCausalLM, 
    processor: AutoProcessor, 
    system_prompt: str, 
    user_prompt: str, 
    max_tokens: int = MAX_TOKENS
) -> Tuple[str, float]:
    """
    Tạo response từ model cho một input đơn lẻ.
    
    Args:
        model: Loaded language model
        processor: Model processor/tokenizer
        system_prompt: System instruction prompt
        user_prompt: User query prompt
        max_tokens: Maximum tokens to generate
        
    Returns:
        Tuple of (json_response, generation_time_seconds)
        
    Raises:
        Exception: Logs error and returns fallback response
        
    Note:
        Uses deterministic generation (do_sample=False) for reproducibility.
        Fixed configuration conflicts by removing sampling parameters.
    """
    messages = []
    if system_prompt and system_prompt.strip():
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})

    try:
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        input_len = inputs["input_ids"].shape[-1]

        start_time = time.time()
        with torch.inference_mode(), torch.autocast(device_type="cuda"):
            generation_output = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=DO_SAMPLE,  
                pad_token_id=processor.pad_token_id
            )
        generation_time = time.time() - start_time
        
        generated_ids = generation_output[0][input_len:]
        response = processor.decode(generated_ids, skip_special_tokens=True)

    except Exception as e:
        print(f"Error in generate_prediction: {str(e)}")
        response = "{}"
        generation_time = 0

    json_response = extract_json_from_response(response)
    return json_response, generation_time

def evaluate_model_on_dataset(
    model: AutoModelForCausalLM, 
    processor: AutoProcessor, 
    dataset: List[Dict[str, Any]], 
    output_file: str, 
    batch_size: int = BATCH_SIZE
) -> List[Dict[str, Any]]:
    """
    Evaluate model on the entire dataset and save results.
    
    Args:
        model: Loaded language model
        processor: Model processor/tokenizer
        dataset: List of dataset samples
        output_file: Path to save results JSON
        batch_size: Batch size for processing
        
    Returns:
        List of processed results with predictions
        
    Note:
        Processes in batches for memory efficiency on 2xT4 setup.
        Includes comprehensive error handling per batch.
    """
    results = []
    total_time = 0
    processed_samples_count = 0
    print(f"Processing {len(dataset)} samples in batches of {batch_size}...")

    system_prompt_content, _ = create_zero_shot_prompt("", "")

    for i in tqdm(range(0, len(dataset), batch_size)):
        batch_samples = dataset[i:i+batch_size]
        if not batch_samples: 
            continue

        batch_texts = [sample.get("text", "") for sample in batch_samples]
        batch_sent_ids = [sample.get("sent_id", f"unknown_id_{i+idx}") for idx, sample in enumerate(batch_samples)]

        batch_messages_for_template = []
        for text_idx, text_content in enumerate(batch_texts):
            current_sent_id = batch_sent_ids[text_idx]
            _, user_prompt_content = create_zero_shot_prompt(text_content, current_sent_id)
            
            current_messages = []
            if system_prompt_content and system_prompt_content.strip():
                 current_messages.append({"role": "system", "content": system_prompt_content})
            current_messages.append({"role": "user", "content": user_prompt_content})
            batch_messages_for_template.append(current_messages)

        try:
            start_time_batch = time.time()
            
            inputs = processor.apply_chat_template(
                batch_messages_for_template,
                add_generation_prompt=True,
                tokenize=True,
                return_tensors="pt",
                padding=True, 
                return_dict=True
            ).to(model.device)

            with torch.inference_mode(), torch.autocast(device_type="cuda"):
                generated_outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_TOKENS,  # ✅ Use config instead of 512
                    do_sample=DO_SAMPLE,        # ✅ Use config instead of False
                    pad_token_id=processor.pad_token_id
                )
            batch_gen_time = time.time() - start_time_batch
            total_time += batch_gen_time
            
            output_ids = generated_outputs[:, inputs.input_ids.shape[1]:]
            batch_responses_raw = processor.batch_decode(output_ids, skip_special_tokens=True)

            for idx, raw_response in enumerate(batch_responses_raw):
                original_text = batch_texts[idx]
                sent_id = batch_sent_ids[idx]
                json_response_str = extract_json_from_response(raw_response)
                result_json_processed = postprocess_response(json_response_str, original_text, sent_id)
                try:
                    result_dict = json.loads(result_json_processed)
                    results.append(result_dict)
                except json.JSONDecodeError:
                     print(f"Failed to parse final JSON for sent_id {sent_id}. Raw: {result_json_processed}")
                     results.append({"sent_id": sent_id, "text": original_text, "opinions": [], "error": "json_parse_error"})
                processed_samples_count += 1
                
        except Exception as e:
            print(f"Error processing batch starting with sample ID {batch_sent_ids[0] if batch_sent_ids else 'N/A'}: {str(e)}")
            for idx_err in range(len(batch_samples)):
                results.append({
                    "sent_id": batch_sent_ids[idx_err] if idx_err < len(batch_sent_ids) else f"error_unknown_id_{i+idx_err}",
                    "text": batch_texts[idx_err] if idx_err < len(batch_texts) else "Unknown text due to error",
                    "opinions": [],
                    "error": str(e)
                })
                processed_samples_count += 1
        
        # ✅ GPU memory cleanup after every 10 batches
        if (i // batch_size + 1) % CLEANUP_FREQUENCY == 0:
            gpu_memory_cleanup()
    
    # ✅ Final cleanup
    gpu_memory_cleanup()
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    avg_time = total_time / processed_samples_count if processed_samples_count > 0 else 0
    print(f"Evaluation complete. Results saved to {output_file}")
    print(f"Total time: {total_time:.2f}s, Processed samples: {processed_samples_count}, Average time per sample: {avg_time:.2f}s")
    return results

def main() -> None:
    """
    Main execution function - Auto dataset evaluation.
    
    Automatically runs inference on dataset with default parameters.
    No user interaction required for streamlined workflow.
    """
    print("🚀 Starting Qwen 2.5 automatic dataset evaluation...")
    print(f"📁 Dataset: {DATASET_PATH}")      # ✅ Use config
    print(f"💾 Output: {OUTPUT_FILE}")        # ✅ Use config  
    print(f"📦 Batch size: {BATCH_SIZE}")     # ✅ Use config
    
    # Initialize model
    print("\n🔧 Initializing model...")
    model, processor = setup_model()
    print("✅ Model ready!")

    # ✅ Prepare experiment config using config variables
    experiment_config = {
        "model_id": MODEL_ID,
        "dataset_path": DATASET_PATH,
        "batch_size": BATCH_SIZE,
        "max_tokens": MAX_TOKENS,
        "do_sample": DO_SAMPLE,
        "dtype": str(TORCH_DTYPE),
        "device_map": DEVICE_MAP,
        "trust_remote_code": TRUST_REMOTE_CODE,
        "experiment_name": EXPERIMENT_NAME
    }

    # Load dataset
    print(f"\n📥 Loading dataset from {DATASET_PATH}...")  # ✅ Use config
    dataset = load_dataset(DATASET_PATH)                   # ✅ Use config
    if not dataset:
        print("❌ Dataset empty or not found. Exiting.")
        return
        
    print(f"✅ Loaded {len(dataset)} samples")
    
    # Run evaluation
    print(f"\n🏃 Starting evaluation...")
    start_time = time.time()
    results = evaluate_model_on_dataset(
        model, processor, dataset, 
        OUTPUT_FILE,    # ✅ Use config
        batch_size=BATCH_SIZE  # ✅ Use config
    )
    total_experiment_time = time.time() - start_time
    
    # Prepare results summary
    results_summary = {
        "total_samples": len(dataset),
        "processed_samples": len(results),
        "total_time_seconds": total_experiment_time,
        "avg_time_per_sample": total_experiment_time / len(results) if results else 0,
        "output_file": OUTPUT_FILE,  # ✅ Use config
        "success_rate": len([r for r in results if "error" not in r]) / len(results) if results else 0
    }
    
    # Save experiment metadata
    print("\n💾 Saving experiment metadata...")
    save_experiment_metadata(experiment_config, results_summary)
    
    # Final summary
    print("\n" + "="*60)
    print("🎉 QWEN 2.5 EVALUATION COMPLETED!")
    print("="*60)
    print(f"📊 Processed: {results_summary['processed_samples']}/{results_summary['total_samples']} samples")
    print(f"⏱️  Total time: {results_summary['total_time_seconds']:.2f}s")
    print(f"🚀 Avg time/sample: {results_summary['avg_time_per_sample']:.2f}s")
    print(f"✅ Success rate: {results_summary['success_rate']:.2%}")
    print(f"💾 Results saved: {OUTPUT_FILE}")  # ✅ Use config
    print("="*60)

In [ ]:
if __name__ == "__main__":
    main()

## Tester

In [ ]:
class ModelTester:
    """
    Optimized testing framework for multiple inference tests without model reload.
    Shows detailed comparison between raw responses and processed results.
    """
    
    def __init__(self):
        self.model = None
        self.processor = None
        self.is_initialized = False
        
    def initialize_model(self):
        """Initialize model once for multiple tests."""
        if not self.is_initialized:
            print("🔧 Initializing model...")
            self.model, self.processor = setup_model()
            self.is_initialized = True
            print("✅ Model ready for testing!")
        else:
            print("✅ Model already initialized!")
    
    def test_single_sample(self, text: str, sent_id: str, show_details: bool = True):
        """
        Test inference on single sample with detailed output comparison.
        
        Args:
            text: Input text to analyze
            sent_id: Sample identifier
            show_details: Whether to show detailed breakdown
            
        Returns:
            Dict with all processing stages
        """
        if not self.is_initialized:
            self.initialize_model()
            
        print(f"\n🧪 TESTING SAMPLE: {sent_id}")
        print("="*60)
        print(f"📝 Input text: '{text}'")
        
        # Create prompts
        system_prompt, user_prompt = create_zero_shot_prompt(text, sent_id)
        
        # STAGE 1: Generate raw response
        print(f"\n🚀 Stage 1: Generating raw response...")
        start_time = time.time()
        try:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
            
            inputs = self.processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            ).to(self.model.device)

            input_len = inputs["input_ids"].shape[-1]
            
            with torch.inference_mode(), torch.autocast(device_type="cuda"):
                generation = self.model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=False  # Deterministic for testing
                )
                generation = generation[0][input_len:]
                
            raw_response = self.processor.decode(generation, skip_special_tokens=True)
            gen_time = time.time() - start_time
            
            print(f"✅ Generation successful! Time: {gen_time:.2f}s")
            
        except Exception as e:
            print(f"❌ Generation failed: {str(e)}")
            return {"error": str(e)}
        
        # STAGE 2: Extract JSON
        print(f"\n🔍 Stage 2: Extracting JSON...")
        json_extracted = extract_json_from_response(raw_response)
        print(f"✅ JSON extraction complete!")
        
        # STAGE 3: Postprocess (add positions, validate format)
        print(f"\n🛠️  Stage 3: Postprocessing...")
        final_result = postprocess_response(json_extracted, text, sent_id)
        print(f"✅ Postprocessing complete!")
        
        # DISPLAY RESULTS
        if show_details:
            self._display_detailed_results(raw_response, json_extracted, final_result, gen_time)
            
        return {
            "raw_response": raw_response,
            "json_extracted": json_extracted,
            "final_result": final_result,
            "generation_time": gen_time,
            "success": True
        }
    
    def _display_detailed_results(self, raw_response, json_extracted, final_result, gen_time):
        """Display detailed comparison of processing stages."""
        print(f"\n📊 DETAILED RESULTS COMPARISON:")
        print("="*80)
        
        print(f"\n🔸 STAGE 1 - RAW MODEL RESPONSE:")
        print("-" * 40)
        print(f"Length: {len(raw_response)} characters")
        print(f"Content preview: {raw_response[:200]}...")
        if len(raw_response) > 200:
            print(f"... (+{len(raw_response)-200} more characters)")
            
        print(f"\n🔸 STAGE 2 - EXTRACTED JSON:")
        print("-" * 40)
        print(f"Length: {len(json_extracted)} characters")
        try:
            import json
            parsed = json.loads(json_extracted)
            print(f"Valid JSON: ✅ {len(parsed.get('opinions', []))} opinions found")
        except:
            print("Valid JSON: ❌ Invalid JSON format")
        print(f"Content: {json_extracted}")
        
        print(f"\n🔸 STAGE 3 - FINAL PROCESSED RESULT:")
        print("-" * 40)
        try:
            import json
            final_parsed = json.loads(final_result)
            print(f"✅ Final JSON valid")
            print(f"📍 Sent ID: {final_parsed.get('sent_id', 'N/A')}")
            print(f"📄 Text: {final_parsed.get('text', 'N/A')}")
            print(f"💭 Opinions: {len(final_parsed.get('opinions', []))}")
            
            for i, opinion in enumerate(final_parsed.get('opinions', [])):
                print(f"   Opinion {i+1}:")
                print(f"     Source: {opinion.get('Source', [[], []])}")
                print(f"     Target: {opinion.get('Target', [[], []])}")
                print(f"     Expression: {opinion.get('Polar_expression', [[], []])}")
                print(f"     Polarity: {opinion.get('Polarity', 'N/A')}")
                print(f"     Intensity: {opinion.get('Intensity', 'N/A')}")
        except Exception as e:
            print(f"❌ Final JSON invalid: {str(e)}")
            
        print(f"\n⏱️  PERFORMANCE: {gen_time:.2f}s generation time")
        print("="*80)
    
    def test_multiple_samples(self, samples: list):
        """Test multiple samples efficiently."""
        if not self.is_initialized:
            self.initialize_model()
            
        print(f"\n🧪 TESTING {len(samples)} SAMPLES")
        print("="*60)
        
        results = []
        for i, (text, sent_id) in enumerate(samples):
            print(f"\n📋 Sample {i+1}/{len(samples)}")
            result = self.test_single_sample(text, sent_id, show_details=True)
            results.append(result)
            
        # Summary
        successful = sum(1 for r in results if r.get('success', False))
        print(f"\n📊 BATCH TEST SUMMARY:")
        print(f"✅ Successful: {successful}/{len(samples)}")
        print(f"❌ Failed: {len(samples)-successful}/{len(samples)}")
        
        return results
    
    def cleanup(self):
        """Clean up GPU memory."""
        if self.is_initialized:
            gpu_memory_cleanup()
            print("🧹 GPU memory cleanup completed")
        

In [ ]:
TEST_SAMPLES = [
    ("Bữa nay ít chửi, nói chuyện nhẹ nhàng hơn, livechim duyên dáng nc vs fan vui tính, ăn mặc sexy vs chất hơn 👌👌 Ủng hộ chị 🔥", "test_001"),
    ("Có mấy thằng cộng nô lên mạng đọc báo lề phải cũng tin răm rắp", "test_002"),
    ("Mai thầy nhớ lập kênh mới ở đâu để bọn em điểm danh nhé", "test_003"),
    ("làm chết người không phân biệt được cố ý và vô ý à, chưa kể tình tiết thế nào m biết không mà phán thế ?", "test_004")
]
tester = ModelTester()